# 042: Python Sets — Complete A–Z Method Reference, Hashing Internals, and Frozen Sets

## 1. WHAT IS A PYTHON SET?

A Python `set` is an **unordered, mutable collection of unique, hashable elements**. It is implemented as a **hash table** (open addressing with linear probing) in CPython.

### CPython Internal Representation (`setobject.h`)
```c
typedef struct {
    PyObject_HEAD
    Py_ssize_t fill;          // Active entries + dummy entries (from deletions)
    Py_ssize_t used;          // Number of active entries (what len() returns)
    Py_ssize_t mask;          // Table size - 1 (table size is always a power of 2)
    setentry *table;          // Array of setentry structs
    setentry smalltable[8];   // Initial inline table (avoids malloc for small sets)
} PySetObject;
```
- **Hash collision resolution**: CPython uses **open addressing** with a **perturbation-based probing** sequence. When a collision occurs at slot `i`, the next slot checked is:
  $$j = (5j + 1 + \text{perturb}) \bmod \text{table\_size}; \quad \text{perturb} \mathrel{>>=} 5$$
- **Load factor**: When the table is more than 2/3 full (`fill * 3 >= (mask + 1) * 2`), CPython resizes the table to the next power of 2.
- **Uniqueness enforcement**: Before inserting, CPython checks if an element with the same hash AND equal value already exists. If so, the insertion is silently skipped.

---


In [ ]:
import sys

print("=" * 70)
print("SECTION 1: Set Memory and Internals")
print("=" * 70)

# Memory growth as elements are added
s = set()
prev = sys.getsizeof(s)
print(f"{'Size':>6} | {'Bytes':>6} | Event")
print("-" * 40)
print(f"{'0':>6} | {prev:>6} | Empty set")

for i in range(30):
    s.add(i)
    curr = sys.getsizeof(s)
    if curr != prev:
        print(f"{len(s):>6} | {curr:>6} | RESIZE (table expanded)")
        prev = curr



---
## 2. COMPLETE A–Z METHOD REFERENCE

---
### 2.1 `set.add(elem)`
- **What it does**: Adds `elem` to the set. If `elem` is already present, does nothing.
- **Returns**: `None`.
- **Time**: $O(1)$ average, $O(n)$ worst case (hash collision chains or table resize).
- **Requires**: `elem` must be **hashable** (`__hash__` defined, not a list/dict/set).


In [ ]:
print("\n" + "=" * 70)
print("METHOD: set.add(elem)")
print("=" * 70)

s = {1, 2, 3}
s.add(4)
print(f"add(4):                      {s}")

s.add(3)  # Already exists — no change
print(f"add(3) — duplicate:          {s}")

# Edge case: adding None
s.add(None)
print(f"add(None):                   {s}")

# Edge case: unhashable element
try:
    s.add([1, 2])
except TypeError as e:
    print(f"add([1,2]):                  TypeError: {e}")

# Can add tuples (hashable) but not lists
s.add((10, 20))
print(f"add((10,20)):                {s}")



### 2.2 `set.remove(elem)` vs `set.discard(elem)`
- `remove(elem)`: Removes `elem`. Raises `KeyError` if not found.
- `discard(elem)`: Removes `elem`. Does **nothing** if not found (no error).
- **Time**: $O(1)$ average.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: set.remove(elem) vs set.discard(elem)")
print("=" * 70)

s = {1, 2, 3, 4, 5}
s.remove(3)
print(f"remove(3):                   {s}")

# remove raises KeyError
try:
    s.remove(99)
except KeyError as e:
    print(f"remove(99):                  KeyError: {e}")

# discard is safe
s.discard(99)  # No error
print(f"discard(99) — safe:          {s}")
s.discard(4)
print(f"discard(4):                  {s}")



### 2.3 `set.pop()`
- **What it does**: Removes and returns an **arbitrary** element.
- **Raises**: `KeyError` if the set is empty.
- **Time**: $O(1)$.
- **Note**: The element removed is NOT guaranteed to be the "first" or "last" — sets are unordered.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: set.pop()")
print("=" * 70)

s = {10, 20, 30, 40}
popped = s.pop()
print(f"pop() returned:              {popped}, set now: {s}")

# Edge case: pop from empty set
try:
    set().pop()
except KeyError as e:
    print(f"pop() on empty set:          KeyError: {e}")



### 2.4 `set.clear()`
- Removes all elements. $O(n)$.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: set.clear()")
print("=" * 70)

s = {1, 2, 3}
s.clear()
print(f"clear():                     {s}")



### 2.5 `set.copy()`
- Returns a **shallow copy**. $O(n)$.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: set.copy()")
print("=" * 70)

original = {1, 2, 3}
copied = original.copy()
copied.add(4)
print(f"original: {original}, copied: {copied}")
print(f"original is copied:          {original is copied}")



---
## 3. SET ALGEBRA OPERATIONS

These are the mathematical set theory operations implemented in Python.

### 3.1 `set.union(*others)` / `|`
- **Returns**: New set with elements from the set AND all others.
- **Time**: $O(n_1 + n_2 + \ldots)$.


In [ ]:
print("\n" + "=" * 70)
print("SET ALGEBRA: union, intersection, difference, symmetric_difference")
print("=" * 70)

A = {1, 2, 3, 4}
B = {3, 4, 5, 6}
C = {5, 6, 7, 8}

# Union
print(f"A | B:                       {A | B}")
print(f"A.union(B, C):               {A.union(B, C)}")



### 3.2 `set.intersection(*others)` / `&`
- **Returns**: New set with elements common to the set AND all others.
- **Time**: $O(\min(n_1, n_2, \ldots))$ — iterates over the smallest set.


In [ ]:
# Intersection
print(f"A & B:                       {A & B}")
print(f"A.intersection(B):           {A.intersection(B)}")



### 3.3 `set.difference(*others)` / `-`
- **Returns**: New set with elements in the set but NOT in others.


In [ ]:
# Difference
print(f"A - B:                       {A - B}")
print(f"A.difference(B):             {A.difference(B)}")



### 3.4 `set.symmetric_difference(other)` / `^`
- **Returns**: New set with elements in EITHER set but NOT in both (XOR).


In [ ]:
# Symmetric difference
print(f"A ^ B:                       {A ^ B}")
print(f"A.symmetric_difference(B):   {A.symmetric_difference(B)}")



### 3.5 In-Place Variants: `update`, `intersection_update`, `difference_update`, `symmetric_difference_update`
- These mutate the set in-place instead of creating a new set.


In [ ]:
print("\n" + "=" * 70)
print("IN-PLACE SET OPERATIONS")
print("=" * 70)

# update (|=)
s = {1, 2}
s.update([3, 4], {5, 6})
print(f"update([3,4], {{5,6}}):        {s}")

# intersection_update (&=)
s = {1, 2, 3, 4, 5}
s.intersection_update({2, 3, 4})
print(f"intersection_update({{2,3,4}}): {s}")

# difference_update (-=)
s = {1, 2, 3, 4, 5}
s.difference_update({3, 4})
print(f"difference_update({{3,4}}):     {s}")

# symmetric_difference_update (^=)
s = {1, 2, 3}
s.symmetric_difference_update({2, 3, 4})
print(f"symmetric_diff_update({{2,3,4}}): {s}")



### 3.6 Subset and Superset Tests
- `issubset(other)` / `<=`: Every element of the set is in `other`.
- `issuperset(other)` / `>=`: Every element of `other` is in the set.
- `isdisjoint(other)`: Returns `True` if the sets share NO elements.
- **Proper subset**: `<` (subset but not equal).


In [ ]:
print("\n" + "=" * 70)
print("SUBSET / SUPERSET / DISJOINT TESTS")
print("=" * 70)

X = {1, 2, 3}
Y = {1, 2, 3, 4, 5}
Z = {6, 7, 8}

print(f"X.issubset(Y):               {X.issubset(Y)}")
print(f"X <= Y:                      {X <= Y}")
print(f"X < Y (proper subset):       {X < Y}")
print(f"Y.issuperset(X):             {Y.issuperset(X)}")
print(f"X.isdisjoint(Z):             {X.isdisjoint(Z)}")  # No common elements
print(f"X.isdisjoint(Y):             {X.isdisjoint(Y)}")  # Has common elements



---
## 4. FROZENSET — THE IMMUTABLE SET

- `frozenset` is the immutable counterpart of `set`.
- It is **hashable** and can be used as a dictionary key or set element.
- It has all the read-only set operations but NO mutation methods (`add`, `remove`, `pop`, `clear`, `update`, etc.).


In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: Frozenset")
print("=" * 70)

fs = frozenset([1, 2, 3, 4])
print(f"frozenset:                   {fs}")
print(f"hash(frozenset):             {hash(fs)}")

# Frozenset as dict key
lookup = {frozenset({1, 2}): "pair", frozenset({3}): "single"}
print(f"Dict with frozenset keys:    {lookup}")

# Frozenset as set element
nested = {frozenset({1, 2}), frozenset({3, 4})}
print(f"Set of frozensets:           {nested}")

# Cannot mutate
try:
    fs.add(5)
except AttributeError as e:
    print(f"frozenset.add():             AttributeError: {e}")

# Set operations still work (return new frozenset)
print(f"fs | {{5, 6}}:                 {fs | frozenset({5, 6})}")
print(f"fs & {{2, 3, 5}}:              {fs & frozenset({2, 3, 5})}")



---
## 5. HASHING MECHANICS AND COLLISION ANALYSIS

- Python's `hash()` returns an integer. The set uses this to compute the table slot: `slot = hash(elem) & mask`.
- **Collision**: When two different elements hash to the same slot, CPython probes the next slot using the perturbation formula.
- **Worst case**: If ALL elements hash to the same slot, lookup degrades to $O(n)$.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: Hash Collision Demonstration")
print("=" * 70)

# Custom class that forces hash collisions
class BadHash:
    def __init__(self, value):
        self.value = value
    def __hash__(self):
        return 42  # All instances hash to the same value!
    def __eq__(self, other):
        return isinstance(other, BadHash) and self.value == other.value
    def __repr__(self):
        return f"BH({self.value})"

import time

# Normal hashing (diverse hashes)
normal = set()
t0 = time.perf_counter()
for i in range(5000):
    normal.add(i)
t1 = time.perf_counter()
normal_time = (t1 - t0) * 1000

# Colliding hashes
colliding = set()
t0 = time.perf_counter()
for i in range(5000):
    colliding.add(BadHash(i))
t1 = time.perf_counter()
collision_time = (t1 - t0) * 1000

print(f"Normal hashing (5K inserts):  {normal_time:.2f} ms")
print(f"All-colliding (5K inserts):   {collision_time:.2f} ms")
print(f"Collision slowdown:           {collision_time / (normal_time + 0.001):.1f}x")



---
## 6. SET COMPREHENSIONS AND PRACTICAL PATTERNS


In [ ]:
print("\n" + "=" * 70)
print("SECTION 6: Set Comprehensions & Practical Patterns")
print("=" * 70)

# Set comprehension
squares = {x*x for x in range(10)}
print(f"Set comprehension squares:   {sorted(squares)}")

# Remove duplicates from a list while preserving order
def deduplicate_ordered(seq):
    seen = set()
    result = []
    for item in seq:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

data = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
print(f"Deduplicated (ordered):      {deduplicate_ordered(data)}")

# Find common elements between multiple lists
list_a = [1, 2, 3, 4, 5]
list_b = [4, 5, 6, 7, 8]
list_c = [5, 6, 7, 8, 9]
common = set(list_a) & set(list_b) & set(list_c)
print(f"Common to all 3 lists:       {common}")



---
## 7. FAANG & INTERVIEW QUESTIONS

### Q1: Why can't you add a list to a set?
**A**: Sets use hash tables internally. Every element must be hashable (i.e., `__hash__` must return a consistent integer). Lists are mutable, so they could change after being added, invalidating their hash. Python prevents this by making lists unhashable. Use tuples instead.

### Q2: What is the time complexity of `x in set` vs `x in list`?
**A**: `x in set` is $O(1)$ average (hash table lookup). `x in list` is $O(n)$ (linear scan). For membership testing, sets are dramatically faster.

### Q3: How does CPython handle hash collisions in sets?
**A**: CPython uses **open addressing** with a **perturbation-based probing** sequence. When slot `i` is occupied, it computes the next probe: `j = (5*j + 1 + perturb) % table_size` with `perturb >>= 5`. This distributes collisions more evenly than simple linear probing.

### Q4: What is the difference between `set.remove()` and `set.discard()`?
**A**: `remove()` raises `KeyError` if the element is not found. `discard()` silently does nothing. Use `discard()` when you don't know if the element exists and don't want to handle exceptions.

### Q5: When would you use a frozenset?
**A**: When you need an immutable, hashable set: (1) as a dictionary key, (2) as an element of another set, (3) to prevent accidental mutation, (4) for thread-safe read-only collections.


In [ ]:
# Membership test speed comparison
print("\n" + "=" * 70)
print("SECTION 7: Set vs List Membership Speed")
print("=" * 70)

import time

N = 100_000
test_list = list(range(N))
test_set = set(range(N))
target = N - 1  # Worst case for list (last element)

# List lookup
t0 = time.perf_counter()
for _ in range(100):
    _ = target in test_list
t1 = time.perf_counter()
list_time = (t1 - t0) * 1000

# Set lookup
t0 = time.perf_counter()
for _ in range(100):
    _ = target in test_set
t1 = time.perf_counter()
set_time = (t1 - t0) * 1000

print(f"'in' list (100 lookups):     {list_time:.2f} ms")
print(f"'in' set (100 lookups):      {set_time:.4f} ms")
print(f"Set is {list_time / (set_time + 0.0001):.0f}x faster for membership testing")
